In [1]:
import pandas as pd
import numpy as np
import textwrap
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv('bbc_text_cls.csv')

In [3]:
df.head()

,text,labels
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business


In [4]:
doc = df[df.labels == 'business']['text'].sample(random_state = 42)

In [5]:
def wrap(x):
  return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

In [6]:
print(wrap(doc.iloc[0]))

Christmas sales worst since 1981

UK retail sales fell in December,
failing to meet expectations and making it by some counts the worst
Christmas since 1981.

Retail sales dropped by 1% on the month in
December, after a 0.6% rise in November, the Office for National
Statistics (ONS) said.  The ONS revised the annual 2004 rate of growth
down from the 5.9% estimated in November to 3.2%. A number of
retailers have already reported poor figures for December.  Clothing
retailers and non-specialist stores were the worst hit with only
internet retailers showing any significant growth, according to the
ONS.

The last time retailers endured a tougher Christmas was 23 years
previously, when sales plunged 1.7%.

The ONS echoed an earlier
caution from Bank of England governor Mervyn King not to read too much
into the poor December figures.  Some analysts put a positive gloss on
the figures, pointing out that the non-seasonally-adjusted figures
showed a performance comparable with 2003. The Novembe

In [7]:
sents = nltk.sent_tokenize(doc.iloc[0].split("\n", 1)[1])

In [8]:
featurizer = TfidfVectorizer(
    stop_words=stopwords.words('english'),
    norm='l1',
)

In [9]:
X = featurizer.fit_transform(sents)

In [10]:
def get_sentence_score(tfidf_row):
    # return the average of the non-zero values
    # of the tf-idf vector representation of a sentence
    x = tfidf_row[tfidf_row != 0]
    return x.mean()

In [11]:
scores = np.zeros(len(sents))
for i in range(len(sents)):
    score = get_sentence_score(X[i, :])
    scores[i] = score

In [12]:
print(scores)

[0.07142857 0.08333333 0.125      0.14285714 0.07692308 0.09090909
 0.07142857 0.07692308 0.06666667 0.07142857 0.125      0.08333333
 0.1        0.06666667 0.07142857 0.04545455 0.1       ]


In [13]:
# Exercise: can you think of a way to compute all the scores in
# one step?
scores = 1.0 / X.getnnz(axis=1)

print("Vectorized scores:", scores)

Vectorized scores: [0.07142857 0.08333333 0.125      0.14285714 0.07692308 0.09090909
 0.07142857 0.07692308 0.06666667 0.07142857 0.125      0.08333333
 0.1        0.06666667 0.07142857 0.04545455 0.1       ]


In [14]:
# Many options for how to choose which sentences to include:

# 1) top N sentences
# 2) top N words or characters.
# 3) top X% sentences or top X% words
# 4) sentences with scores > average score
# 5) sentences with scores > factor * average score

# You also don't have to sort. May make more sense in order.
sort_idx = np.argsort(scores)[::-1]
N = 3
print(f"--- Generated summary (Top {N} sentences) ---")
for i in sort_idx[:N]:
    print(wrap("%.2f: %s" % (scores[i], sents[i])))
print("\n")
avg_score = scores.mean()
print("--- Generated summary (Above Average Score) ---")
for i in range(len(sents)):
    if scores[i] > avg_score:
        print(wrap("%.2f: %s" % (scores[i], sents[i])))

--- Generated summary (Top 3 sentences) ---
0.14: A number of retailers have already reported poor figures for
December.
0.12: The ONS revised the annual 2004 rate of growth down from the
5.9% estimated in November to 3.2%.
0.12: However, reports from some High Street retailers highlight the
weakness of the sector.


--- Generated summary (Above Average Score) ---
0.12: The ONS revised the annual 2004 rate of growth down from the
5.9% estimated in November to 3.2%.
0.14: A number of retailers have already reported poor figures for
December.
0.09: The last time retailers endured a tougher Christmas was 23 years
previously, when sales plunged 1.7%.
0.12: However, reports from some High Street retailers highlight the
weakness of the sector.
0.10: And a British Retail Consortium survey found that Christmas 2004
was the worst for 10 years.
0.10: "Our view is the Bank of England will keep its powder dry and
wait to see the big picture."


In [15]:
doc.iloc[0].split("\n", 1)[0]

'Christmas sales worst since 1981'

In [16]:
def summarize(text):
    # extract sentences
    sents = nltk.sent_tokenize(text)

    # perform tf-idf
    X = featurizer.fit_transform(sents)

    # compute scores for each sentence
    scores = np.zeros(len(sents))
    for i in range(len(sents)):
        score = get_sentence_score(X[i, :])
        scores[i] = score
# sort the scores
    sort_idx = np.argsort(-scores)

    # print summary
    for i in sort_idx[:5]:
        print(wrap("%.2f: %s" % (scores[i], sents[i])))

In [17]:
doc = df[df.labels == 'entertainment']['text'].sample(random_state=123)
summarize(doc.iloc[0].split("\n", 1)[1])

0.11: The Black Eyed Peas won awards for best R 'n' B video and
sexiest video, both for Hey Mama.
0.10: The ceremony was held at the Luna Park fairground in Sydney
Harbour and was hosted by the Osbourne family.
0.10: The VH1 First Music Award went to Cher honouring her
achievements within the music industry.
0.10: Goodrem, Green Day and the Black Eyed Peas took home two awards
each.
0.10: Other winners included Green Day, voted best group, and the
Black Eyed Peas.


In [18]:
doc.iloc[0].split("\n", 1)[0]

'Goodrem wins top female MTV prize'

In [19]:
print(wrap(doc.iloc[0]))

Goodrem wins top female MTV prize

Pop singer Delta Goodrem has
scooped one of the top individual prizes at the first Australian MTV
Music Awards.

The 21-year-old singer won the award for best female
artist, with Australian Idol runner-up Shannon Noll taking the title
of best male at the ceremony.  Goodrem, known in both Britain and
Australia for her role as Nina Tucker in TV soap Neighbours, also
performed a duet with boyfriend Brian McFadden.  Other winners
included Green Day, voted best group, and the Black Eyed Peas.
Goodrem, Green Day and the Black Eyed Peas took home two awards each.
As well as best female, Goodrem also took home the Pepsi Viewers
Choice Award, whilst Green Day bagged the prize for best rock video
for American Idiot.  The Black Eyed Peas won awards for best R 'n' B
video and sexiest video, both for Hey Mama.  Local singer and
songwriter Missy Higgins took the title of breakthrough artist of the
year, with Australian Idol winner Guy Sebastian taking the honours f

![](https://deeplearningcourses.com/notebooks_v3_pxl?sc=N3y7GKxUH16CxQS403JQ2A&n=Text+Summarization)